# CMPE492 — Final Evaluation (3 configs + significance test)

**One-shot notebook.** Scores all three configurations and runs the paired significance test in a single run:
- **Hybrid-LP** (our method: MimicTalk→LivePortrait)
- **LP-only** (controlled baseline)
- **Wav2Lip** (external SOTA baseline)

Metrics per identity: LSE-C, LSE-D, AV offset (SyncNet) + CSIM (ArcFace).
Then: mean±std aggregation + paired Wilcoxon significance tests.

**Inputs:**
- `batch_eval.zip` (hybrid_lp/, lp_only/, references/)
- `wav2lip_videos.zip` (id01.mp4 ... id10.mp4)

---
**Run order:** 1a → (RESTART) → 1b → 2 → 3 → 4 → 5 → 6

## Cell 1a — numpy/scipy fix (run, then RESTART SESSION)

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'numpy==1.26.4', 'scipy==1.13.1', '--force-reinstall'], check=True)
print('✅ Done — NOW: Runtime → Restart session, then run Cell 1b')

## Cell 1b — Setup SyncNet + ArcFace (after restart)

In [ ]:
import os, sys, re, json, subprocess, shlex, shutil, glob, urllib.request
import numpy as np, cv2, torch

SYNCNET_DIR = '/content/syncnet_python'
WORK_DIR    = '/tmp/syncnet_work'

print('=== SyncNet Setup ===')
if not os.path.exists(SYNCNET_DIR) or not os.path.exists(f'{SYNCNET_DIR}/run_pipeline.py'):
    if os.path.exists(SYNCNET_DIR): shutil.rmtree(SYNCNET_DIR)
    subprocess.run(f'git clone https://github.com/joonson/syncnet_python.git {SYNCNET_DIR}',
                   shell=True, check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'scenedetect[opencv]', 'python-speech-features'], capture_output=True, check=True)
os.makedirs(f'{SYNCNET_DIR}/data', exist_ok=True)
for fname, url in [
    ('syncnet_v2.model', 'https://www.robots.ox.ac.uk/~vgg/software/lipsync/data/syncnet_v2.model'),
    ('sfd_face.pth', 'https://www.robots.ox.ac.uk/~vgg/software/lipsync/data/sfd_face.pth')]:
    dst = f'{SYNCNET_DIR}/data/{fname}'
    if not os.path.exists(dst): urllib.request.urlretrieve(url, dst)
s3fd_dst = f'{SYNCNET_DIR}/detectors/s3fd/weights/sfd_face.pth'
os.makedirs(os.path.dirname(s3fd_dst), exist_ok=True)
if not os.path.exists(s3fd_dst): shutil.copy(f'{SYNCNET_DIR}/data/sfd_face.pth', s3fd_dst)
for fname in glob.glob(f'{SYNCNET_DIR}/**/*.py', recursive=True):
    with open(fname) as f: content = f.read()
    orig = content
    content = re.sub(r'\bnp\.int\b(?!6|3|1|_|e)', 'np.int64', content)
    content = re.sub(r'\bnp\.float\b(?!6|3|1|_|i)', 'np.float64', content)
    content = re.sub(r'\bnp\.bool\b(?!_)', 'bool', content)
    if content != orig:
        with open(fname, 'w') as f: f.write(content)
print('  ✅ SyncNet ready')

if SYNCNET_DIR in sys.path: sys.path.remove(SYNCNET_DIR)
sys.path.insert(0, SYNCNET_DIR)
import importlib.util
for mod_name in ['SyncNetModel', 'SyncNetInstance']:
    spec = importlib.util.spec_from_file_location(mod_name, f'{SYNCNET_DIR}/{mod_name}.py')
    m = importlib.util.module_from_spec(spec); sys.modules[mod_name] = m
    spec.loader.exec_module(m)
SyncNetInstance = sys.modules['SyncNetInstance'].SyncNetInstance
syncnet = SyncNetInstance()
syncnet.loadParameters(f'{SYNCNET_DIR}/data/syncnet_v2.model')
print('  ✅ SyncNet model loaded')

print('\n=== ArcFace Setup ===')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'insightface', 'onnxruntime-gpu'], capture_output=True)
from insightface.app import FaceAnalysis
prov = ['CUDAExecutionProvider'] if torch.cuda.is_available() else ['CPUExecutionProvider']
arcface = FaceAnalysis(name='buffalo_l', providers=prov)
arcface.prepare(ctx_id=0 if 'CUDA' in prov[0] else -1, det_size=(512, 512))
arcface.det_model.det_thresh = 0.3
print('  ✅ ArcFace ready')
print('\n✅ Cell 1b complete')

## Cell 2 — Upload Inputs
> Upload `batch_eval.zip` AND `wav2lip_videos.zip`

In [ ]:
import os, zipfile, glob
from google.colab import files

BATCH = '/content/final_eval'
os.makedirs(BATCH, exist_ok=True)

print('Upload batch_eval.zip AND wav2lip_videos.zip')
uploaded = files.upload()
for fname, data in uploaded.items():
    zpath = f'/content/{fname}'
    with open(zpath, 'wb') as f: f.write(data)
    if 'batch_eval' in fname.lower():
        with zipfile.ZipFile(zpath) as z: z.extractall(BATCH)
        print(f'✅ {fname} → final_eval/')
    elif 'wav2lip' in fname.lower():
        os.makedirs(f'{BATCH}/wav2lip', exist_ok=True)
        with zipfile.ZipFile(zpath) as z: z.extractall(f'{BATCH}/wav2lip')
        print(f'✅ {fname} → wav2lip/')

def find_videos(pattern):
    out = {}
    for v in sorted(glob.glob(pattern, recursive=True)):
        stem = os.path.splitext(os.path.basename(v))[0]
        out[stem] = v
    return out

CONFIGS = {
    'hybrid_lp': find_videos(f'{BATCH}/**/hybrid_lp/*.mp4'),
    'lp_only':   find_videos(f'{BATCH}/**/lp_only/*.mp4'),
    'wav2lip':   find_videos(f'{BATCH}/wav2lip/**/*.mp4'),
}
references = find_videos(f'{BATCH}/**/references/*.png')

print('\n── Discovered ──')
for cfg, v in CONFIGS.items():
    print(f'  {cfg}: {len(v)} → {sorted(v.keys())}')
print(f'  references: {len(references)}')
common = sorted(set(CONFIGS['hybrid_lp']) & set(CONFIGS['lp_only']) & set(CONFIGS['wav2lip']))
print(f'\n  Common to all 3 configs: {len(common)} → {common}')

## Cell 3 — Scoring Functions

In [ ]:
import argparse

def score_syncnet(video_path, ref_name):
    eval_clip = f'/tmp/eval_{ref_name}.mp4'
    probe = subprocess.run(
        f'ffprobe -v quiet -show_entries stream=codec_type -of csv=p=0 "{video_path}"',
        shell=True, capture_output=True, text=True).stdout
    ff = ['ffmpeg','-y','-i',video_path]
    if 'audio' not in probe:
        ff += ['-f','lavfi','-i','anullsrc=r=16000:cl=mono','-map','0:v:0','-map','1:a:0']
    ff += ['-t','15','-c:v','libx264','-pix_fmt','yuv420p','-c:a','aac','-ac','1','-ar','16000','-shortest',eval_clip]
    subprocess.run(ff, capture_output=True)
    for d in ['pyavi','pytmp','pycrop','pywork']:
        p = f'{WORK_DIR}/{d}/{ref_name}'
        if os.path.exists(p): shutil.rmtree(p)
    os.makedirs(WORK_DIR, exist_ok=True)
    subprocess.run(
        f'cd {SYNCNET_DIR} && python run_pipeline.py --videofile {shlex.quote(eval_clip)} '
        f'--reference {ref_name} --data_dir {WORK_DIR} --min_track 25',
        shell=True, capture_output=True, text=True)
    crop_files = sorted(glob.glob(f'{WORK_DIR}/pycrop/{ref_name}/*.avi'))
    if not crop_files: return None, None, None
    saved = os.getcwd(); os.chdir(SYNCNET_DIR)
    opt = argparse.Namespace(initial_model='data/syncnet_v2.model', batch_size=20, vshift=15,
        data_dir=WORK_DIR, avi_dir=f'{WORK_DIR}/pyavi', tmp_dir=f'{WORK_DIR}/pytmp',
        work_dir=f'{WORK_DIR}/pywork/{ref_name}', crop_dir=f'{WORK_DIR}/pycrop', reference=ref_name)
    offs, confs, dists = [], [], []
    for cf in crop_files:
        try:
            o,c,d = syncnet.evaluate(opt, videofile=cf)
            offs.append(o); confs.append(c); dists.append(float(np.mean(d)))
        except Exception as e: print(f'      ⚠ {e}')
    os.chdir(saved)
    if confs: return float(np.mean(confs)), float(np.mean(dists)), float(np.mean(offs))
    return None, None, None

def biggest(faces):
    return max(faces, key=lambda f:(f.bbox[2]-f.bbox[0])*(f.bbox[3]-f.bbox[1])) if faces else None

def score_csim(video_path, ref_image_path):
    ref = cv2.imread(ref_image_path)
    if ref is None: return None
    ref = cv2.resize(ref,(512,512))
    rf = biggest(arcface.get(ref))
    if rf is None: return None
    re_ = rf.embedding.astype(np.float32); re_ /= np.linalg.norm(re_)+1e-8
    cap = cv2.VideoCapture(video_path); sims=[]
    while True:
        ok, fr = cap.read()
        if not ok: break
        f = biggest(arcface.get(fr))
        if f:
            e = f.embedding.astype(np.float32); e/=np.linalg.norm(e)+1e-8
            sims.append(float(np.dot(re_,e)))
    cap.release()
    return float(np.mean(sims)) if sims else None

print('✅ Scoring functions ready')

## Cell 4 — Score All 3 Configs

In [ ]:
import time
results = {cfg: {} for cfg in CONFIGS}
t_start = time.time()
total = sum(len(CONFIGS[c]) for c in CONFIGS if c)
done = 0
for cfg in CONFIGS:
    print(f'\n{"="*55}\n  {cfg}\n{"="*55}')
    for stem in common:
        done += 1
        vpath = CONFIGS[cfg][stem]
        ref_name = f'{cfg}_{stem}'.replace('-','_')
        lse_c, lse_d, av = score_syncnet(vpath, ref_name)
        csim = score_csim(vpath, references[stem]) if stem in references else None
        results[cfg][stem] = {'lse_c':lse_c,'lse_d':lse_d,'av_offset':av,'csim':csim}
        print(f'  [{done}/{len(common)*3}] {cfg}/{stem}: LSE-C={lse_c}, LSE-D={lse_d}, AV={av}, CSIM={csim}')
print(f'\n✅ Scoring complete in {time.time()-t_start:.0f}s')

## Cell 5 — Aggregate + Paired Significance Tests

In [ ]:
import numpy as np
from scipy.stats import wilcoxon

def vals(cfg, metric):
    return [results[cfg][s][metric] for s in common if results[cfg][s].get(metric) is not None]

def stat(cfg, metric):
    a = np.array(vals(cfg, metric), dtype=float)
    return (a.mean(), a.std()) if len(a) else (None, None)

METRICS = ['lse_c','lse_d','av_offset','csim']
print('='*80)
print('  CMPE492 — Final Multi-Identity Benchmark (mean ± std, n={})'.format(len(common)))
print('='*80)
print(f'  {"Config":<12} {"LSE-C↑":>14} {"LSE-D↓":>14} {"AV off":>12} {"CSIM↑":>14}')
print('-'*80)
for cfg in CONFIGS:
    row = f'  {cfg:<12}'
    for m in METRICS:
        mu, sd = stat(cfg, m)
        d = 4 if m=='csim' else (1 if m=='av_offset' else 3)
        row += f' {(f"{mu:.{d}f}±{sd:.{d}f}" if mu is not None else "N/A"):>14}'
    print(row)
print('='*80)

# ── Paired significance tests (per-identity, matched) ──
print('\n── Paired Wilcoxon signed-rank tests (LSE-C and CSIM) ──')
def paired_test(cfgA, cfgB, metric):
    pairs = [(results[cfgA][s][metric], results[cfgB][s][metric])
             for s in common
             if results[cfgA][s].get(metric) is not None and results[cfgB][s].get(metric) is not None]
    if len(pairs) < 5:
        return None
    a = np.array([p[0] for p in pairs]); b = np.array([p[1] for p in pairs])
    try:
        stat_, p = wilcoxon(a, b)
        return {'n':len(pairs), 'meanA':float(a.mean()), 'meanB':float(b.mean()),
                'p_value':float(p), 'significant': p < 0.05}
    except Exception as e:
        return {'error': str(e)}

comparisons = [
    ('hybrid_lp','lp_only','lse_c'),
    ('hybrid_lp','wav2lip','lse_c'),
    ('hybrid_lp','lp_only','csim'),
    ('hybrid_lp','wav2lip','csim'),
]
sig_results = {}
for a,b,m in comparisons:
    r = paired_test(a,b,m)
    sig_results[f'{a}_vs_{b}__{m}'] = r
    if r and 'p_value' in r:
        verdict = '✅ SIGNIFICANT' if r['significant'] else 'not significant'
        print(f'  {a} vs {b} [{m}]: {r["meanA"]:.3f} vs {r["meanB"]:.3f}, '
              f'p={r["p_value"]:.4f} ({verdict})')
    else:
        print(f'  {a} vs {b} [{m}]: {r}')

## Cell 6 — Export

In [ ]:
import json, csv
from datetime import datetime, timezone

OUT = '/content/final_eval/results'
os.makedirs(OUT, exist_ok=True)

agg = {}
for cfg in CONFIGS:
    agg[cfg] = {}
    for m in METRICS:
        mu, sd = stat(cfg, m)
        agg[cfg][m] = {'mean': mu, 'std': sd}

full = {'per_identity': results, 'aggregates': agg, 'significance': sig_results,
        'configs': list(CONFIGS.keys()), 'n_identities': len(common),
        'timestamp': datetime.now(timezone.utc).isoformat()}
with open(f'{OUT}/final_results.json','w') as f: json.dump(full, f, indent=2)
print(f'✅ {OUT}/final_results.json')

with open(f'{OUT}/final_per_identity.csv','w',newline='') as f:
    w = csv.writer(f)
    w.writerow(['config','identity','lse_c','lse_d','av_offset','csim'])
    for cfg in CONFIGS:
        for s in common:
            r = results[cfg][s]
            w.writerow([cfg,s,r['lse_c'],r['lse_d'],r['av_offset'],r['csim']])
print(f'✅ {OUT}/final_per_identity.csv')

from google.colab import files
files.download(f'{OUT}/final_results.json')
files.download(f'{OUT}/final_per_identity.csv')
print('\n✅ Final evaluation complete — send me final_results.json and we finish the report')